## 依赖注入


如果这段逻辑满足`「多个地方要用」`+`「与具体业务无关」`+`「失败就该中断请求」`三条中的两条以上，就抽成依赖。




### 1. 核心思想


使用依赖注入来共享通用逻辑，避免代码重复



In [ ]:
from fastapi import FastAPI, Query,Depends
app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello World"}

# 分页参数逻辑共用：商品列表和用户列表
def common_params(
    skip: int = Query(0, ge=0),
    limit: int = Query(10, le=60)
) -> dict:
    return {"skip": skip, "limit": limit}

@app.get("/item/item_list")
def get_item_list(commons: dict = Depends(common_params)):
    return {"commons": commons}

@app.get("/user/user_list")
def get_user_list(commons: dict = Depends(common_params)):
    return {"commons": commons}

`Depends` 做了什么：当请求到达时，FastAPI 先调用 `common_params`，拿到返回值，然后把它注入到 `get_xxx_list` 的 `commons` 参数中

### 2. 依赖注入写法

函数依赖

In [ ]:
async def common_parameters(q: str = None, skip: int = 0, limit: int = 100):
    return {"q": q, "skip": skip, "limit": limit}

@app.get("/items/")
async def read_items(commons: dict = Depends(common_parameters)):
    return commons

类依赖

In [ ]:
class CommonQueryParams:
    def __init__(self, q: str = None, skip: int = 0, limit: int = 100):
        self.q = q
        self.skip = skip
        self.limit = limit

@app.get("/items/")
async def read_items(commons: CommonQueryParams = Depends()):
    return commons

路径操作装饰器依赖

In [ ]:
from fastapi import HTTPException, Header
async def verify_token(x_token: str = Header(...)):
    if x_token != "fake-super-secret-token":
        raise HTTPException(status_code=400, detail="X-Token header invalid")

@app.get("/items/", dependencies=[Depends(verify_token)])
async def read_items():
    return [{"item": "Foo"}, {"item": "Bar"}]

### 3. 依赖注入应用场景
- 处理请求参数：从请求中提取参数和验证参数（路径参数、查询参数、请求体）
- 共享业务逻辑：抽取封装多个路由公用的逻辑代码
- 共享数据库连接：管理数据库会话的创建、使用、关闭
- 安全和认证：验证用户身份、检查权限和角色要求等

| # | 场景 | 依赖形态 | 有无返回值 | 挂载位置 | 说明 |
|---|---|---|---|---|---|
| 1 | 数据库会话 | `yield` 函数 | 有（Session） | 参数级 | 最经典，`finally` 保证 close |
| 2 | 身份认证 | 函数 + Security | 有（User） | 参数级 / router 级 | 失败抛 401 |
| 3 | 权限校验 | 闭包工厂 | 有（User） | 参数级 | `require_role("admin")` |
| 4 | 可选鉴权 | 函数（参数可 None） | `User \| None` | 参数级 | 登录态可选的公开接口 |
| 5 | 分页 / 排序 / 过滤 | 类或函数 | 有（PageParams） | 参数级 | 让路由签名保持干净 |
| 6 | 多租户 / 数据隔离 | 函数 | 有（tenant_id） | router 级 | 全模块强制带租户条件 |
| 7 | 审计日志 / 限流 / 埋点 | 函数 | 无 | `dependencies=[...]` | 纯副作用，当轻量中间件用 |
| 8 | 外部客户端 / 配置 | 工厂 + lifespan | 有（httpx.AsyncClient） | 参数级 | 连接池在 lifespan 建，依赖只借用 |


